---

Created for [Pricing and Hedging Derivative Securities: Theory and Methods](https://book.derivative-securities.org/)

Authored by
- Kerry Back, Rice University
- Hong Liu, Washington University in St. Louis
- Mark Loewenstein, University of Maryland
 
---

<a target="_blank" href="https://colab.research.google.com/github/math-finance-book/book-code/blob/main/17_AppendixBinomial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
import plotly.graph_objects as go

def plot_symmetric_binomial_tree(S0=100, u=1.1):
    d = 1 / u  # Down factor
    periods = 3  # Number of periods

    # Define nodes with symmetric positioning
    nodes = {}
    for t in range(periods + 1):
        for j in range(t + 1):
            x = t  # Time step on x-axis
            y = 2 * j - t  # Centered y-axis positioning for symmetry
            nodes[(x, y)] = round(S0 * (u ** j) * (d ** (t - j)), 2)

    # Define edges
    edges = []
    for t in range(periods):
        for j in range(t + 1):
            x = t
            y = 2 * j - t
            edges.append(((x, y), (x + 1, y + 1)))  # Up move
            edges.append(((x, y), (x + 1, y - 1)))  # Down move

    fig = go.Figure()

    # Add edges to the plot
    for edge in edges:
        x_coords = [edge[0][0], edge[1][0]]
        y_coords = [edge[0][1], edge[1][1]]
        fig.add_trace(go.Scatter(x=x_coords, y=y_coords, mode='lines', line=dict(color='black'), showlegend=False))

    # Add nodes to the plot
    x_vals = [key[0] for key in nodes.keys()]
    y_vals = [key[1] for key in nodes.keys()]
    labels = [str(nodes[key]) for key in nodes.keys()]

    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals, mode='markers+text',
        marker=dict(size=20, color='lightblue'),
        text=labels, textposition="top center",
        showlegend=False
    ))

    fig.update_layout(
        title="3-Period Symmetric Binomial Tree (u=1.1, d=1/u)",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='white',
        width=700,
        height=500
    )

    fig.show()

plot_symmetric_binomial_tree()

In [ ]:

import matplotlib.pyplot as plt

def plot_symmetric_binomial_tree(S0=100, u=1.1, vertical_scale=0.5, title_pad=20):
    """
    Plots a 3-period symmetric binomial tree using Matplotlib,
    with options to compress the vertical spacing and add space between the title and the plot.

    :param S0: initial stock price
    :param u: up factor
    :param vertical_scale: compresses or expands the vertical spacing
    :param title_pad: extra spacing between the title and the tree (in points)
    """
    d = 1 / u  # Down factor
    periods = 3  # Number of periods

    # Define nodes with symmetric positioning
    nodes = {}
    for t in range(periods + 1):
        for j in range(t + 1):
            x = t
            y = vertical_scale * (2 * j - t)  # scale the vertical distance
            nodes[(x, y)] = round(S0 * (u**j) * (d**(t - j)), 2)

    # Define edges
    edges = []
    for t in range(periods):
        for j in range(t + 1):
            x = t
            y = vertical_scale * (2 * j - t)
            # Up move
            edges.append(((x, y), (x + 1, y + vertical_scale)))
            # Down move
            edges.append(((x, y), (x + 1, y - vertical_scale)))

    # Create a Matplotlib figure
    fig, ax = plt.subplots(figsize=(7, 5))

    # Plot edges
    for edge in edges:
        x_coords = [edge[0][0], edge[1][0]]
        y_coords = [edge[0][1], edge[1][1]]
        ax.plot(x_coords, y_coords, color='black')

    # Plot nodes
    x_vals = [k[0] for k in nodes.keys()]
    y_vals = [k[1] for k in nodes.keys()]
    labels = [str(nodes[k]) for k in nodes.keys()]
    ax.scatter(x_vals, y_vals, s=300, color='lightblue')

    # Add text labels above each node
    for x, y, label in zip(x_vals, y_vals, labels):
        ax.text(x, y + 0.3 * vertical_scale, label, ha='center', va='bottom')

    # Title with extra spacing set by 'pad'
    ax.set_title(
        f"3-Period Symmetric Binomial Tree (u={u}, d={round(d,2)})",
        pad=title_pad
    )

    # Style the plot
    ax.axis("equal")
    ax.axis("off")

    plt.show()

# Example usage: extra spacing of 20 points between the caption and the tree
plot_symmetric_binomial_tree(vertical_scale=0.5, title_pad=20)


In [ ]:
import plotly.graph_objects as go
from collections import defaultdict

def plot_variable_branching_tree_symmetric():
    # 1. Build tree nodes/edges
    nodes = {(0, 0): 100}  # Root node at time t=0, index=0 => value=100
    edges = []

    # next_index[t] tracks how many nodes have been created at period t so far
    next_index = defaultdict(int)
    next_index[0] = 1  # We have 1 node at t=0

    # transitions[(t, i)] = list of factors for each branch out of node (t, i)
    transitions = {
        (0, 0): [1.1, 0.9],       # At t=0, i=0 => 2 branches
        (1, 0): [1.2, 1.0, 0.8],  # At t=1, i=0 => 3 branches
        (1, 1): [1.1, 0.85],      # At t=1, i=1 => 2 branches
        (2, 0): [1.3, 0.9],       # At t=2, i=0 => 2 branches
        (2, 1): [1.2, 1.0],       # At t=2, i=1 => 2 branches
        (2, 2): [1.1],            # At t=2, i=2 => 1 branch
        (2, 3): [1.15, 0.95, 0.85], # t=2, i=3 => 3 branches
        (2, 4): [1.1, 0.9]        # t=2, i=4 => 2 branches
    }

    # Create nodes/edges
    for (t, i), factors in transitions.items():
        base_value = nodes[(t, i)]
        for factor in factors:
            child_t = t + 1
            child_i = next_index[child_t]
            next_index[child_t] += 1

            child_value = round(base_value * factor, 2)
            nodes[(child_t, child_i)] = child_value

            edges.append(((t, i), (child_t, child_i)))

    # 2. Count how many nodes per period => assign symmetrical y-coordinates
    nodes_in_period = defaultdict(list)
    for (t, i), val in nodes.items():
        nodes_in_period[t].append(i)

    # For each period t, sort node indexes, then map them to symmetrical positions around 0
    coords = {}  # coords[(t, i)] = y_position
    for t in sorted(nodes_in_period.keys()):
        node_list = sorted(nodes_in_period[t])
        count = len(node_list)

        # We'll assign positions from 0..(count-1), then shift so center is 0
        for idx, node_i in enumerate(node_list):
            # E.g., if count=5, positions -> 0,1,2,3,4 => shift by -2 => -2,-1,0,1,2
            shift = -(count - 1) / 2
            y = idx + shift
            coords[(t, node_i)] = y

    # 3. Build the plot
    fig = go.Figure()

    # Add edges
    for ((t1, i1), (t2, i2)) in edges:
        x_coords = [t1, t2]
        y_coords = [coords[(t1, i1)], coords[(t2, i2)]]
        fig.add_trace(go.Scatter(
            x=x_coords,
            y=y_coords,
            mode='lines',
            line=dict(color='black'),
            showlegend=False
        ))

    # Add nodes (markers+text)
    for (t, i), val in nodes.items():
        fig.add_trace(go.Scatter(
            x=[t],
            y=[coords[(t, i)]],
            mode='markers+text',
            marker=dict(size=10, color='blue'),
            text=[str(val)],
            textposition='top center',
            showlegend=False
        ))

    fig.update_layout(
        title="3-Period Variable Branching Tree (Symmetric Layout)",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        width=900,
        height=600,
        plot_bgcolor='white'
    )

    fig.show()

# Call the plotting function
plot_variable_branching_tree_symmetric()


In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_variable_branching_tree_scaled(vertical_scale=0.5, horizontal_scale=2.0):
    """
    Plots a 3-period variable branching tree in a narrower vertical 
    layout and a longer horizontal layout.

    :param vertical_scale: Factor (<1 = narrower, >1 = taller) for vertical spacing
    :param horizontal_scale: Factor (>1 = longer, <1 = shorter) for horizontal spacing
    """
    # 1. Build tree nodes/edges
    nodes = {(0, 0): 100}  # Root node at time t=0, index=0 => value=100
    edges = []

    # Track how many nodes created at each period t
    next_index = defaultdict(int)
    next_index[0] = 1

    # transitions: dict[(t, i)] -> list of factors from node (t,i)
    transitions = {
        (0, 0): [1.1, 0.9],
        (1, 0): [1.2, 1.0, 0.8],
        (1, 1): [1.1, 0.85],
        (2, 0): [1.3, 0.9],
        (2, 1): [1.2, 1.0],
        (2, 2): [1.1],
        (2, 3): [1.15, 0.95, 0.85],
        (2, 4): [1.1, 0.9]
    }

    # Create nodes/edges
    for (t, i), factors in transitions.items():
        base_val = nodes[(t, i)]
        for factor in factors:
            child_t = t + 1
            child_i = next_index[child_t]
            next_index[child_t] += 1

            child_val = round(base_val * factor, 2)
            nodes[(child_t, child_i)] = child_val
            edges.append(((t, i), (child_t, child_i)))

    # 2. Assign symmetrical y-coords for each period
    nodes_in_period = defaultdict(list)
    for (t, i) in nodes.keys():
        nodes_in_period[t].append(i)

    coords = {}
    for t in sorted(nodes_in_period.keys()):
        node_list = sorted(nodes_in_period[t])
        count = len(node_list)
        # positions => 0..(count-1), shift so center is 0
        for idx, node_i in enumerate(node_list):
            shift = -(count - 1) / 2
            y = (idx + shift) * vertical_scale
            coords[(t, node_i)] = y

    # 3. Build the Matplotlib figure
    fig, ax = plt.subplots(figsize=(12, 4))  # Wider figure

    # Plot edges
    for ((t1, i1), (t2, i2)) in edges:
        x_coords = [t1 * horizontal_scale, t2 * horizontal_scale]
        y_coords = [coords[(t1, i1)], coords[(t2, i2)]]
        ax.plot(x_coords, y_coords, color='black')

    # Plot nodes + text
    for (t, i), val in nodes.items():
        x_ = t * horizontal_scale
        y_ = coords[(t, i)]
        ax.scatter(x_, y_, s=100, color='blue')
        ax.text(x_, y_ + 0.2 * vertical_scale, str(val),
                ha='center', va='bottom')

    ax.set_title("3-Period Variable Branching Tree (Scaled)")
    ax.axis("equal")
    ax.axis("off")
    plt.show()

# Example usage: narrower by vertical_scale=0.5, longer by horizontal_scale=2.0
plot_variable_branching_tree_scaled(vertical_scale=0.5, horizontal_scale=2.0)